# Demand Forecasting Model

## Load Data

In [7]:
import pandas as pd
from pathlib import Path
import os

PROCESSED_DIR = Path("../data/processed")


In [9]:
# Load all datasets
orders = pd.read_parquet(PROCESSED_DIR / "orders_clean.parquet")
calendar = pd.read_parquet(PROCESSED_DIR / "calendar_clean.parquet")
inventory = pd.read_parquet(PROCESSED_DIR / "inventory_clean.parquet")
reviews = pd.read_parquet(PROCESSED_DIR / "reviews_clean.parquet")

In [10]:
orders.head()

,order_id,order_date,day_of_week,product_category,occasion_tag,quantity_sold,revenue,order_channel,delivery_flag
0,ORD-00001,2023-01-01,Sunday,orchids,corporate,6,216.54,in-store,False
1,ORD-00002,2023-01-01,Sunday,tulips,walk-in,1,13.32,in-store,True
2,ORD-00003,2023-01-01,Sunday,hydrangeas,corporate,6,177.70,online,False
3,ORD-00004,2023-01-01,Sunday,roses,corporate,6,91.35,online,True
4,ORD-00005,2023-01-01,Sunday,lilies,anniversary,5,109.99,online,True


In [11]:
calendar.head()

,date,holiday_name,days_until_next_major_holiday,is_university_event_week,is_weekend,season,avg_temp_f,precipitation_inches,is_holiday
0,2023-01-01,None,44,False,True,winter,32,0.63,False
1,2023-01-02,None,43,False,False,winter,29,0.00,False
2,2023-01-03,None,42,False,False,winter,33,0.00,False
3,2023-01-04,None,41,False,False,winter,38,0.00,False
4,2023-01-05,None,40,False,False,winter,29,0.11,False


In [12]:
inventory.head()

,order_week,product_category,units_ordered,units_sold,units_wasted,unit_cost,unit_price,restock_lead_time_days
0,2022-12-26,gardenias,4,4,0,6.53,17.18,3
1,2022-12-26,hydrangeas,7,7,0,10.45,26.12,3
2,2022-12-26,lilies,4,4,0,7.96,19.91,3
3,2022-12-26,orchids,5,5,0,18.19,40.42,3
4,2022-12-26,roses,5,5,0,5.11,13.46,3


In [13]:
reviews.head()

,review_id,platform,review_date,star_rating,review_text,occasion_mentioned,review_text_clean
0,REV-001,google,2023-01-01,5,Love this little shop on Mass Ave. Always stop...,walk-in,love this little shop on mass ave. always stop...
1,REV-002,yelp,2023-01-01,5,Just moved to the area and this place is a hid...,walk-in,just moved to the area and this place is a hid...
2,REV-003,google,2023-01-03,4,Quick birthday bouquet pickup. In and out in 1...,None,quick birthday bouquet pickup. in and out in 1...
3,REV-004,yelp,2023-01-05,1,Ordered a specific color scheme for a birthday...,None,ordered a specific color scheme for a birthday...
4,REV-005,yelp,2023-01-08,2,Was browsing and the staff member hovering ove...,walk-in,was browsing and the staff member hovering ove...


## Further Feature Engineering

In [20]:
# Transform orders dataframe
orders["week_start"] = (orders["order_date"] - pd.to_timedelta(pd.to_datetime(orders["order_date"]).dt.weekday, unit="D")).apply(pd.to_datetime).dt.normalize()
orders["order_channel_enc"] = (orders["order_channel"] == "online").astype(int)
orders["delivery_flag_enc"] = orders["delivery_flag"].astype(int)

orders.head()



,order_id,order_date,day_of_week,product_category,occasion_tag,quantity_sold,revenue,order_channel,delivery_flag,order_channel_enc,delivery_flag_enc,week_start
0,ORD-00001,2023-01-01,Sunday,orchids,corporate,6,216.54,in-store,False,0,0,2022-12-26
1,ORD-00002,2023-01-01,Sunday,tulips,walk-in,1,13.32,in-store,True,0,1,2022-12-26
2,ORD-00003,2023-01-01,Sunday,hydrangeas,corporate,6,177.70,online,False,1,0,2022-12-26
3,ORD-00004,2023-01-01,Sunday,roses,corporate,6,91.35,online,True,1,1,2022-12-26
4,ORD-00005,2023-01-01,Sunday,lilies,anniversary,5,109.99,online,True,1,1,2022-12-26


In [23]:
# Fill in holiday_name None values
calendar["holiday_name"] = calendar["holiday_name"].fillna("none")

# Convert 'date' to datetime if not already, and get 'week_start' (Monday)
calendar["date"] = pd.to_datetime(calendar["date"])
calendar["week_start"] = (calendar["date"] - pd.to_timedelta(calendar["date"].dt.weekday, unit="D")).dt.normalize()

# Aggregate daily to weekly
def mode(series):
    """Return the most frequent value (mode); if tie, returns the first."""
    return series.mode().iloc[0]

def holiday_name_selector(names: pd.Series) -> str:
    """Pick first non-'none' holiday_name in the week, or 'none' if absent."""
    non_none = names[names != "none"]
    if not non_none.empty:
        return non_none.iloc[0]
    return "none"

calendar_weekly = (
    calendar
    .drop(columns=["is_weekend"])
    .groupby("week_start", as_index=False)
    .agg({
        "holiday_name": holiday_name_selector,
        "days_until_next_major_holiday": "min",
        "is_university_event_week": "max",
        "is_holiday": "max",
        "season": mode,
        "avg_temp_f": "mean",
        "precipitation_inches": "sum",
    })
)

calendar.head()

,date,holiday_name,days_until_next_major_holiday,is_university_event_week,is_weekend,season,avg_temp_f,precipitation_inches,is_holiday,week_start
0,2023-01-01,none,44,False,True,winter,32,0.63,False,2022-12-26
1,2023-01-02,none,43,False,False,winter,29,0.00,False,2023-01-02
2,2023-01-03,none,42,False,False,winter,33,0.00,False,2023-01-02
3,2023-01-04,none,41,False,False,winter,38,0.00,False,2023-01-02
4,2023-01-05,none,40,False,False,winter,29,0.11,False,2023-01-02


In [ ]:
# Feature engineering for inventory
inventory["waste_rate"] = inventory["units_wasted"] / inventory["units_ordered"]
inventory["waste_rate"] = inventory["waste_rate"].replace([float("inf"), -float("inf")], 0).fillna(0)

inventory["waste_cost"] = inventory["units_wasted"] * inventory["unit_cost"]

inventory["sell_through_rate"] = inventory["units_sold"] / inventory["units_ordered"]
inventory["sell_through_rate"] = inventory["sell_through_rate"].replace([float("inf"), -float("inf")], 0).fillna(0)

if "order_week" in inventory.columns:
    inventory = inventory.rename(columns={"order_week": "week_start"})

orders.head()


,order_id,order_date,day_of_week,product_category,occasion_tag,quantity_sold,revenue,order_channel,delivery_flag,order_channel_enc,delivery_flag_enc,week_start
0,ORD-00001,2023-01-01,Sunday,orchids,corporate,6,216.54,in-store,False,0,0,2022-12-26
1,ORD-00002,2023-01-01,Sunday,tulips,walk-in,1,13.32,in-store,True,0,1,2022-12-26
2,ORD-00003,2023-01-01,Sunday,hydrangeas,corporate,6,177.70,online,False,1,0,2022-12-26
3,ORD-00004,2023-01-01,Sunday,roses,corporate,6,91.35,online,True,1,1,2022-12-26
4,ORD-00005,2023-01-01,Sunday,lilies,anniversary,5,109.99,online,True,1,1,2022-12-26


In [28]:
import nltk
from nltk.sentiment import SentimentIntensityAnalyzer

# Ensure VADER lexicon is available before running sentiment analysis
try:
    sia = SentimentIntensityAnalyzer()
except LookupError:
    nltk.download("vader_lexicon")
    sia = SentimentIntensityAnalyzer()

# Fill missing 'occasion_mentioned' with "unknown"
reviews["occasion_mentioned"] = reviews["occasion_mentioned"].fillna("unknown")

# Mark positive reviews: star_rating >= 4
reviews["positive"] = (reviews["star_rating"] >= 4).astype(int)

# Run VADER sentiment analysis on review_text_clean
reviews["vader_compound"] = reviews["review_text_clean"].apply(
    lambda t: sia.polarity_scores(t)["compound"]
)

# Assign sentiment label based on compound score
def vader_label(compound: float) -> str:
    if compound >= 0.05:
        return "positive"
    elif compound <= -0.05:
        return "negative"
    return "neutral"

reviews["vader_label"] = reviews["vader_compound"].apply(vader_label)
reviews["sentiment_score"] = reviews["vader_compound"]

# Add week_start - snap review_date to the Monday of its week
reviews["week_start"] = pd.to_datetime(reviews["review_date"]) - pd.to_timedelta(
    pd.to_datetime(reviews["review_date"]).dt.weekday, unit="D"
)
reviews["week_start"] = reviews["week_start"].dt.normalize()

# Aggregate to (week_start, occasion_mentioned)
grouped = (
    reviews.groupby(["week_start", "occasion_mentioned"], observed=True)
    .agg(
        positive_rate=("positive", "mean"),
        avg_sentiment=("sentiment_score", "mean"),
        review_count=("review_id", "count"),
        avg_star=("star_rating", "mean"),
    )
    .reset_index()
)

# Pivot wide per occasion_mentioned
wide = grouped.pivot(
    index="week_start",
    columns="occasion_mentioned",
    values=["positive_rate", "avg_sentiment", "review_count", "avg_star"],
)
# Flatten columns
wide.columns = [
    f"{kind}_{occ}" if kind != "avg_star" else f"stars_{occ}"
    for kind, occ in wide.columns
]
# Sentiment/Review NaN -> 0, star NaN stays NaN
for col in wide.columns:
    if col.startswith(("positive_rate_", "avg_sentiment_", "review_count_")):
        wide[col] = wide[col].fillna(0)

# For clarity, rename positive_rate_* and review_count_* to sentiment_* and reviews_*
wide = wide.rename(
    columns={
        **{c: c.replace("positive_rate_", "sentiment_") for c in wide.columns if c.startswith("positive_rate_")},
        **{c: c.replace("review_count_", "reviews_") for c in wide.columns if c.startswith("review_count_")},
        **{c: c.replace("avg_sentiment_", "avg_sentiment_") for c in wide.columns if c.startswith("avg_sentiment_")}
    }
)

occasion_sentiment_weekly = wide.reset_index()

occasion_sentiment_weekly.head()

,week_start,sentiment_anniversary,sentiment_birthday,sentiment_corporate,sentiment_graduation,sentiment_hospital,sentiment_mother's day,sentiment_sympathy,sentiment_unknown,sentiment_valentine's day,...,stars_birthday,stars_corporate,stars_graduation,stars_hospital,stars_mother's day,stars_sympathy,stars_unknown,stars_valentine's day,stars_walk-in,stars_wedding
0,2022-12-26,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.0,NaN
1,2023-01-02,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.5,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,2.5,NaN,2.0,NaN
2,2023-01-23,0.5,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,4.0,NaN,1.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2023-01-30,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.000000,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,5.0,NaN
4,2023-02-06,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.666667,...,NaN,NaN,NaN,NaN,2.0,NaN,4.5,3.666667,3.0,NaN


## Aggregations

In [29]:

# --- Weekly demand by (week_start, product_category) ---

weekly_demand = (
    orders.groupby(["week_start", "product_category"], observed=True)
    .agg(
        total_qty_sold=("quantity_sold", "sum"),
        total_revenue=("revenue", "sum"),
        order_count=("order_id", "count"),
        online_orders=("order_channel", lambda x: (x == "online").sum()),
        delivery_orders=("delivery_flag", "sum"),
    )
    .reset_index()
)

# Calculate online_share and delivery_share
weekly_demand["online_share"] = (
    weekly_demand["online_orders"] / weekly_demand["order_count"]
).round(3)
weekly_demand["delivery_share"] = (
    weekly_demand["delivery_orders"] / weekly_demand["order_count"]
).round(3)

# Drop raw counts, just keep shares and main metrics
weekly_demand = weekly_demand.drop(columns=["online_orders", "delivery_orders"])

weekly_demand.head()

# --- Weekly occasion pivot ---

# Group by (week_start, occasion_tag) and count orders per occasion
occasion_counts = (
    orders.groupby(["week_start", "occasion_tag"], observed=True)
    .agg(occ_count=("order_id", "count"))
    .reset_index()
)

# Pivot so each occasion_tag is a separate column
weekly_occasion_pivot = occasion_counts.pivot(
    index="week_start",
    columns="occasion_tag",
    values="occ_count",
).fillna(0)

# Make columns names occ_{tag}
weekly_occasion_pivot.columns = [f"occ_{c}" for c in weekly_occasion_pivot.columns]

# Add total and percentage columns
weekly_occasion_pivot["occ_total"] = weekly_occasion_pivot.sum(axis=1)

for tag_col in weekly_occasion_pivot.columns:
    if tag_col.startswith("occ_") and tag_col != "occ_total":
        weekly_occasion_pivot[f"pct_{tag_col[4:]}"] = (
            weekly_occasion_pivot[tag_col] / weekly_occasion_pivot["occ_total"]
        ).fillna(0).round(3)

weekly_occasion_pivot = weekly_occasion_pivot.reset_index()

weekly_occasion_pivot.head()



,week_start,occ_anniversary,occ_birthday,occ_corporate,occ_graduation,occ_hospital,occ_mother's day,occ_sympathy,occ_valentine's day,occ_walk-in,...,pct_anniversary,pct_birthday,pct_corporate,pct_graduation,pct_hospital,pct_mother's day,pct_sympathy,pct_valentine's day,pct_walk-in,pct_wedding
0,2022-12-26,1.0,1.0,3.0,0.0,0.0,0.0,1.0,0.0,3.0,...,0.111,0.111,0.333,0.000,0.000,0.000,0.111,0.000,0.333,0.000
1,2023-01-02,2.0,4.0,8.0,2.0,4.0,0.0,4.0,2.0,15.0,...,0.047,0.093,0.186,0.047,0.093,0.000,0.093,0.047,0.349,0.047
2,2023-01-09,4.0,7.0,3.0,4.0,4.0,1.0,6.0,1.0,21.0,...,0.077,0.135,0.058,0.077,0.077,0.019,0.115,0.019,0.404,0.019
3,2023-01-16,2.0,6.0,8.0,2.0,1.0,0.0,5.0,0.0,15.0,...,0.048,0.143,0.190,0.048,0.024,0.000,0.119,0.000,0.357,0.071
4,2023-01-23,7.0,5.0,4.0,0.0,1.0,0.0,2.0,0.0,9.0,...,0.250,0.179,0.143,0.000,0.036,0.000,0.071,0.000,0.321,0.000


In [31]:
# --- Master Panel Assembly ---

# Left join weekly_demand and weekly_occasion_pivot on week_start
panel = weekly_demand.merge(
    weekly_occasion_pivot, how="left", on="week_start"
)

# Add inventory on (week_start, product_category)
panel = panel.merge(
    inventory,
    how="left",
    on=["week_start", "product_category"],
    suffixes=("", "_inv"),
)

# Add calendar_weekly on week_start
panel = panel.merge(
    calendar_weekly,
    how="left",
    on="week_start",
    suffixes=("", "_cal"),
)

# Load weekly_panel.parquet and merge any missing columns (if any)
from pathlib import Path
import pandas as pd

parquet_path = Path("../data/processed/weekly_panel.parquet")
if parquet_path.exists():
    weekly_panel = pd.read_parquet(parquet_path)
    # Find columns in weekly_panel not already present in panel
    extra_columns = [
        c for c in weekly_panel.columns
        if c not in panel.columns or c == "week_start" or c == "product_category"
    ]
    # Exclude join keys
    extra_columns = [
        c for c in extra_columns if c not in ["week_start", "product_category"]
    ]
    if extra_columns:
        panel = panel.merge(
            weekly_panel[["week_start", "product_category"] + extra_columns],
            how="left",
            on=["week_start", "product_category"]
        )

panel.head()

,week_start,product_category,total_qty_sold,total_revenue,order_count,online_share,delivery_share,occ_anniversary,occ_birthday,occ_corporate,...,revenue,distinct_occasions,is_holiday_week,holiday_names,min_days_to_holiday,total_precipitation_inches,quantity_sold_lag1,quantity_sold_roll4_mean,year,week_of_year
0,2022-12-26,gardenias,4,84.85,1,1.0,1.0,1.0,1.0,3.0,...,84.85,1.0,False,None,44,0.63,NaN,NaN,2022,52
1,2022-12-26,hydrangeas,7,200.09,2,0.5,0.0,1.0,1.0,3.0,...,200.09,2.0,False,None,44,0.63,NaN,NaN,2022,52
2,2022-12-26,lilies,5,109.99,1,1.0,1.0,1.0,1.0,3.0,...,109.99,1.0,False,None,44,0.63,NaN,NaN,2022,52
3,2022-12-26,orchids,6,216.54,1,0.0,0.0,1.0,1.0,3.0,...,216.54,1.0,False,None,44,0.63,NaN,NaN,2022,52
4,2022-12-26,roses,6,91.35,1,1.0,1.0,1.0,1.0,3.0,...,91.35,1.0,False,None,44,0.63,NaN,NaN,2022,52


In [32]:
# Print summary info
print("Master panel shape:", panel.shape)
print("\nColumns:", list(panel.columns))
print("\nNull value counts:")
print(panel.isnull().sum())

Master panel shape: (942, 55)

Columns: ['week_start', 'product_category', 'total_qty_sold', 'total_revenue', 'order_count', 'online_share', 'delivery_share', 'occ_anniversary', 'occ_birthday', 'occ_corporate', 'occ_graduation', 'occ_hospital', "occ_mother's day", 'occ_sympathy', "occ_valentine's day", 'occ_walk-in', 'occ_wedding', 'occ_total', 'pct_anniversary', 'pct_birthday', 'pct_corporate', 'pct_graduation', 'pct_hospital', "pct_mother's day", 'pct_sympathy', "pct_valentine's day", 'pct_walk-in', 'pct_wedding', 'units_ordered', 'units_sold', 'units_wasted', 'unit_cost', 'unit_price', 'restock_lead_time_days', 'waste_rate', 'waste_cost', 'sell_through_rate', 'holiday_name', 'days_until_next_major_holiday', 'is_university_event_week', 'is_holiday', 'season', 'avg_temp_f', 'precipitation_inches', 'quantity_sold', 'revenue', 'distinct_occasions', 'is_holiday_week', 'holiday_names', 'min_days_to_holiday', 'total_precipitation_inches', 'quantity_sold_lag1', 'quantity_sold_roll4_mean', '

## Feature Engineering on Master Dataset

In [34]:
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore", category=FutureWarning)

master_panel = panel.copy()

# --- Holiday Flags ---
def week_flag(df: pd.DataFrame, cond: pd.Series) -> pd.Series:
    """
    For each week_start, flag if any row within that week meets the condition.

    Args:
        df: DataFrame with a "week_start" column.
        cond: Boolean Series (same length as df) with True for rows to flag.

    Returns:
        Series of 0/1 for each row, indicating condition ever met in their week.
    """
    # Ensure cond is a Series with index aligned to df
    if not isinstance(cond, pd.Series):
        raise ValueError("cond must be a pandas Series")
    # Assign week_start index for groupby
    return cond.groupby(df["week_start"]).transform("max").astype(int)

# Is this week Valentine's, Mother's Day, Graduation, or any holiday week?
master_panel["is_valentines_week"] = week_flag(
    master_panel, master_panel["holiday_name"].str.lower() == "valentine's day"
)
master_panel["is_mothers_day_week"] = week_flag(
    master_panel, master_panel["holiday_name"].str.lower() == "mother's day"
)
master_panel["is_graduation_week"] = week_flag(
    master_panel, master_panel["holiday_name"].str.lower() == "graduation"
)
master_panel["is_holiday_week"] = week_flag(
    master_panel, master_panel["is_holiday"] == True
)

# --- Holiday Proximity Bucket ---
proximity_bins = [0, 3, 7, 14, np.inf]
proximity_labels = ["0-3 days", "4-7 days", "8-14 days", "15+"]
master_panel["holiday_proximity"] = pd.cut(
    master_panel["days_until_next_major_holiday"],
    bins=proximity_bins,
    labels=proximity_labels,
    include_lowest=True,
    ordered=True,
)

# --- Lag features (units_sold) ---
master_panel = master_panel.sort_values(["product_category", "week_start"])
master_panel["units_sold_lag_1w"] = (
    master_panel.groupby("product_category", group_keys=False)["units_sold"].shift(1)
)
master_panel["units_sold_lag_4w"] = (
    master_panel.groupby("product_category", group_keys=False)["units_sold"].shift(4)
)
master_panel["units_sold_roll4_mean"] = (
    master_panel.groupby("product_category", group_keys=False)["units_sold"]
    .shift(1)
    .rolling(4, min_periods=1)
    .mean()
    .reset_index(level=0, drop=True)
)

# --- Season one-hot (drop season_winter as reference) ---
season_dummies = pd.get_dummies(master_panel["season"], prefix="season")
season_cols = ["season_spring", "season_summer", "season_fall"]
for col in season_cols:
    master_panel[col] = season_dummies[col] if col in season_dummies else 0

# --- Product one-hot (drop roses as reference) ---
product_dummies = pd.get_dummies(master_panel["product_category"], prefix="product")
product_cols = [col for col in product_dummies.columns if col != "product_roses"]
for col in product_cols:
    master_panel[col] = product_dummies[col]

# --- Column order: [units_sold, week_start, product_category, ...features] ---
core_cols = ["units_sold", "week_start", "product_category"]
other_cols = [c for c in master_panel.columns if c not in core_cols]
features_df = master_panel[core_cols + other_cols]

features_df.head()

,units_sold,week_start,product_category,total_qty_sold,total_revenue,order_count,online_share,delivery_share,occ_anniversary,occ_birthday,...,season_summer,season_fall,product_ferns,product_gardenias,product_hydrangeas,product_lilies,product_orchids,product_peonies,product_sunflowers,product_tulips
7,28,2023-01-02,ferns,31,302.09,5,0.400,0.200,2.0,4.0,...,False,False,True,False,False,False,False,False,False,False
16,15,2023-01-09,ferns,15,173.89,3,0.333,0.333,4.0,7.0,...,False,False,True,False,False,False,False,False,False,False
25,19,2023-01-16,ferns,30,366.51,6,0.000,0.000,2.0,6.0,...,False,False,True,False,False,False,False,False,False,False
34,6,2023-01-23,ferns,6,76.46,2,0.500,0.500,7.0,5.0,...,False,False,True,False,False,False,False,False,False,False
43,14,2023-01-30,ferns,14,175.42,4,0.250,0.000,4.0,6.0,...,False,False,True,False,False,False,False,False,False,False


In [35]:
# --- Summary ---
print("features_df shape:", features_df.shape)
print("\nNull value counts:")
print(features_df.isnull().sum())

features_df shape: (942, 73)

Null value counts:
units_sold            0
week_start            0
product_category      0
total_qty_sold        0
total_revenue         0
                     ..
product_lilies        0
product_orchids       0
product_peonies       0
product_sunflowers    0
product_tulips        0
Length: 73, dtype: int64


**Feature Dataset Columns**:
The feature dataset (`features_df`) includes the following columns, which are constructed to serve as inputs for the demand forecasting model:

- **units_sold**: The target variable — represents the quantity of the product sold in each week/category.
- **week_start**: The start date of the week for each observation, serving as the temporal reference.
- **product_category**: Flower category for the observation; used as a categorical key and for generating product dummies.

**Feature Columns:**
- **Lag features:**  
  - `units_sold_lag_1w`, `units_sold_lag_4w`: Number of units sold by the same product category in the previous 1 and 4 weeks, respectively.  
  - `units_sold_roll4_mean`: Mean sales for the product category over the 4 weeks preceding the current week (excluding the current week).
  These provide recent sales context and help capture autocorrelation in demand.

- **Seasonal one-hot columns:**  
  - `season_spring`, `season_summer`, `season_fall`: Binary indicators for each season using winter as the baseline. These enable the model to account for seasonal demand patterns.

- **Product one-hot columns:**  
  - `product_<category>`: Binary columns indicating each product category, with roses as the reference group. These control for baseline demand differences across product types.

- **Other engineered features:**  
  The dataset may include additional features such as:
    - Holiday/event indicators (e.g., whether week contains/abuts a holiday, or the days until next holiday)
    - Weather data (temperature, precipitation for the week)
    - Channel shares (e.g., online order share, delivery share)
    - Occurrence counts or proportions of purchase occasions (e.g., walk-in, anniversary, birthday)
    - Aggregate review sentiment and rating scores
    - Inventory and restock information
    - Time features (e.g., week of year, year, min days to next holiday)

These features collectively allow the demand forecasting model to learn from time trends, product differences, seasonality, external events, weather, and recent sales behavior. The one-hot columns ensure that categorical information is interpretable by the model, while lagged/rolling sales figures capture temporal dependencies crucial for accurate weekly demand predictions.

# Model Training

In [ ]:
# Merge occasion sentiment features into master panel
features_df = features_df.merge(
    occasion_sentiment_weekly, how="left", on="week_start"
)

# Fill NaN sentiment/review columns with 0 (weeks with no reviews)
sentiment_cols = [c for c in occasion_sentiment_weekly.columns if c != "week_start"]
for col in sentiment_cols:
    if col in features_df.columns:
        features_df[col] = features_df[col].fillna(0)

print("features_df shape after sentiment merge:", features_df.shape)
print("New sentiment columns:", len(sentiment_cols))

In [ ]:
# --- Train / Test Split: 2023 train, 2024 test ---

# Drop rows where lag features are NaN (first week per category)
features_df = features_df.dropna(subset=["units_sold_lag_1w"]).reset_index(drop=True)

# Extract year from week_start
features_df["year"] = features_df["week_start"].dt.year

train_df = features_df[features_df["year"] <= 2023].copy()
test_df = features_df[features_df["year"] == 2024].copy()

print(f"Train: {len(train_df)} rows ({train_df['week_start'].min().date()} to {train_df['week_start'].max().date()})")
print(f"Test:  {len(test_df)} rows ({test_df['week_start'].min().date()} to {test_df['week_start'].max().date()})")

In [ ]:
# --- Define feature columns and target ---

TARGET = "units_sold"

# Columns to exclude from features (target, keys, leaky, redundant)
EXCLUDE_COLS = [
    TARGET, "week_start", "product_category", "year",
    # Leaky / redundant with target
    "total_qty_sold", "total_revenue", "revenue", "quantity_sold",
    "order_count", "units_ordered", "units_wasted",
    "waste_rate", "waste_cost", "sell_through_rate",
    "quantity_sold_lag1", "quantity_sold_roll4_mean",
    "distinct_occasions",
    # Non-numeric / will be encoded separately
    "holiday_name", "holiday_names", "season", "holiday_proximity",
]

FEATURE_COLS = [c for c in features_df.columns if c not in EXCLUDE_COLS]

# Ensure all feature columns are numeric
X_train = train_df[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
y_train = train_df[TARGET].values

X_test = test_df[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
y_test = test_df[TARGET].values

print(f"Features: {len(FEATURE_COLS)} columns")
print(f"X_train: {X_train.shape}, X_test: {X_test.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --- Naive lag-1 baseline ---
naive_pred = test_df["units_sold_lag_1w"].values

# --- Linear Regression ---
lr = LinearRegression()
lr.fit(X_train, y_train)
lr_pred = np.clip(lr.predict(X_test), 0, None)

# --- Histogram Gradient Boosted Tree (default params) ---
hgbt = HistGradientBoostingRegressor(random_state=0, max_iter=300)
hgbt.fit(X_train, y_train)
hgbt_pred = np.clip(hgbt.predict(X_test), 0, None)

# --- Scoring helper ---
def score_model(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    return {"model": name, "MAE": round(mae, 2), "RMSE": round(rmse, 2)}

baseline_results = pd.DataFrame([
    score_model("Naive Lag-1", y_test, naive_pred),
    score_model("Linear Regression", y_test, lr_pred),
    score_model("HGBT (default)", y_test, hgbt_pred),
])

print("=== Baseline Results (2024 hold-out) ===")
baseline_results

## Hyperparameter Tuning (HGBT)

Grid search over HGBT hyperparameters using `TimeSeriesSplit` on the 2023 training data to respect chronological order.

In [ ]:
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# Sort training data chronologically for time-series CV
train_sorted = train_df.sort_values("week_start").reset_index(drop=True)
X_train_sorted = train_sorted[FEATURE_COLS].apply(pd.to_numeric, errors="coerce").fillna(0)
y_train_sorted = train_sorted[TARGET].values

param_grid = {
    "learning_rate": [0.05, 0.1, 0.2],
    "max_depth": [3, 5, None],
    "min_samples_leaf": [10, 20],
    "l2_regularization": [0.0, 1.0],
}

tscv = TimeSeriesSplit(n_splits=5)

grid_search = GridSearchCV(
    HistGradientBoostingRegressor(random_state=0, max_iter=500),
    param_grid,
    cv=tscv,
    scoring="neg_root_mean_squared_error",
    n_jobs=-1,
    verbose=1,
)
grid_search.fit(X_train_sorted, y_train_sorted)

print(f"\nBest CV RMSE: {-grid_search.best_score_:.2f}")
print(f"Best params: {grid_search.best_params_}")

In [ ]:
# Evaluate the tuned HGBT on the 2024 hold-out
tuned_hgbt = grid_search.best_estimator_
tuned_pred = np.clip(tuned_hgbt.predict(X_test), 0, None)

all_results = pd.DataFrame([
    score_model("Naive Lag-1", y_test, naive_pred),
    score_model("Linear Regression", y_test, lr_pred),
    score_model("HGBT (default)", y_test, hgbt_pred),
    score_model("HGBT (tuned)", y_test, tuned_pred),
])

print("=== All Models — Overall (2024 hold-out) ===")
all_results

In [ ]:
# --- Per-category evaluation (tuned HGBT vs naive) ---

categories = test_df["product_category"].values
per_cat_rows = []

for cat in sorted(np.unique(categories)):
    mask = categories == cat
    for name, preds in [("Naive Lag-1", naive_pred), ("Linear Regression", lr_pred),
                         ("HGBT (default)", hgbt_pred), ("HGBT (tuned)", tuned_pred)]:
        row = score_model(name, y_test[mask], preds[mask])
        row["category"] = cat
        per_cat_rows.append(row)

per_cat_results = pd.DataFrame(per_cat_rows)

# Pivot for readability
print("=== MAE per Category (2024 hold-out) ===")
mae_pivot = per_cat_results.pivot(index="category", columns="model", values="MAE")
mae_pivot = mae_pivot[["Naive Lag-1", "Linear Regression", "HGBT (default)", "HGBT (tuned)"]]
print(mae_pivot.to_string())

print("\n=== RMSE per Category (2024 hold-out) ===")
rmse_pivot = per_cat_results.pivot(index="category", columns="model", values="RMSE")
rmse_pivot = rmse_pivot[["Naive Lag-1", "Linear Regression", "HGBT (default)", "HGBT (tuned)"]]
print(rmse_pivot.to_string())

## Diagnostic Plots

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# --- 1. Predicted vs Actual scatter (tuned HGBT) ---
fig, ax = plt.subplots(figsize=(7, 7))
for cat in sorted(test_df["product_category"].unique()):
    mask = (test_df["product_category"] == cat).values
    ax.scatter(y_test[mask], tuned_pred[mask], s=22, alpha=0.7, label=cat)
lim = max(y_test.max(), tuned_pred.max()) * 1.05
ax.plot([0, lim], [0, lim], color="black", linestyle="--", linewidth=1)
ax.set_xlim(0, lim)
ax.set_ylim(0, lim)
ax.set_xlabel("Actual units_sold")
ax.set_ylabel("Predicted units_sold")
ax.set_title("Tuned HGBT: Predicted vs Actual (2024 hold-out)")
ax.legend(fontsize=8, loc="upper left")
fig.tight_layout()
plt.show()

In [ ]:
# --- 2. Residual distribution ---
residuals = y_test - tuned_pred

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(residuals, bins=30, edgecolor="black", alpha=0.7)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_xlabel("Residual (actual - predicted)")
axes[0].set_ylabel("Count")
axes[0].set_title("Residual Distribution — Tuned HGBT")

axes[1].scatter(tuned_pred, residuals, s=15, alpha=0.5)
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_xlabel("Predicted units_sold")
axes[1].set_ylabel("Residual")
axes[1].set_title("Residuals vs Predicted")

fig.tight_layout()
plt.show()

In [ ]:
# --- 3. Feature importance (permutation-based on 2024 test set) ---
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    tuned_hgbt, X_test, y_test, n_repeats=10, random_state=0, n_jobs=-1
)
importances = pd.Series(perm.importances_mean, index=FEATURE_COLS).sort_values(ascending=False)

# Show top 20
fig, ax = plt.subplots(figsize=(9, 7))
importances.head(20).sort_values().plot.barh(ax=ax, color="steelblue")
ax.set_xlabel("Mean decrease in R² (permutation importance)")
ax.set_title("Tuned HGBT — Top 20 Feature Importances (2024 hold-out)")
fig.tight_layout()
plt.show()

print("\nTop 20 features:")
importances.head(20)

In [ ]:
# --- 4. Time-series overlay: actual vs predicted for a sample category ---
sample_cat = "roses"
cat_mask = (test_df["product_category"] == sample_cat).values
weeks = pd.to_datetime(test_df.loc[cat_mask, "week_start"].values)
order = np.argsort(weeks.values)

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(weeks.values[order], y_test[cat_mask][order], marker="o", label="Actual", color="black")
ax.plot(weeks.values[order], tuned_pred[cat_mask][order], marker="s", label="HGBT (tuned)", alpha=0.8)
ax.plot(weeks.values[order], naive_pred[cat_mask][order], marker="^", label="Naive Lag-1", alpha=0.6, linestyle="--")
ax.set_title(f"{sample_cat.title()} — Weekly Demand, 2024 Hold-out")
ax.set_ylabel("units_sold")
ax.set_xlabel("week_start")
ax.legend()
fig.autofmt_xdate()
fig.tight_layout()
plt.show()